# Warm-up 2B · Rice-Leaf Disease Diagnosis · Chẩn đoán bệnh lúa đa phương thức (Multi-modal Classification · Phân loại đa phương thức)

| | 🇬🇧 English | 🇻🇳 Tiếng Việt |
|---|---|---|
| **Task** | A farming app in the Mekong Delta receives reports from farmers. Each report has a **leaf photo**, the farmer's **description in Vietnamese** and **field/weather data**. Predict the `label` of each report (5 classes below). | Một ứng dụng nông nghiệp ở Đồng bằng sông Cửu Long nhận báo cáo từ nông dân. Mỗi báo cáo gồm **ảnh lá lúa**, **mô tả bằng tiếng Việt** của nông dân và **dữ liệu đồng ruộng/thời tiết**. Dự đoán `label` của từng báo cáo (5 lớp bên dưới). |
| **Data** | `train/` 4,000 reports (with label) · `public_test/` 1,000 · `private_test/` 1,000. Each folder has `<split>.csv` and `images/` (96×96 JPG). | `train/` 4.000 báo cáo (có nhãn) · `public_test/` 1.000 · `private_test/` 1.000. Mỗi thư mục có `<split>.csv` và `images/` (ảnh JPG 96×96). |
| **Metric** | Macro-F1 over the 5 classes, **higher is better**. Ranking uses the **private** score. | Macro-F1 trên 5 lớp, **càng cao càng tốt**. Xếp hạng theo điểm **private**. |
| **Submission** | `public_submission.csv`, `private_submission.csv` with columns `id,label` (class names, e.g. `brown_spot`) | `public_submission.csv`, `private_submission.csv` gồm cột `id,label` (tên lớp, vd `brown_spot`) |
| **Rules** | PyTorch model, no external data (pretrained `torchvision` weights are allowed). | Mô hình PyTorch, không dùng dữ liệu ngoài (được dùng trọng số huấn luyện sẵn của `torchvision`). |

$$F_1^{(k)}=\frac{2\,P_k R_k}{P_k+R_k},\qquad \text{Macro-}F_1=\frac{1}{5}\sum_{k=1}^{5}F_1^{(k)}$$

🇬🇧 Every class counts equally, so rare classes matter as much as `healthy`.
🇻🇳 Mọi lớp có trọng số như nhau, nên các lớp hiếm quan trọng không kém lớp `healthy`.

| `label` | 🇬🇧 English | 🇻🇳 Tiếng Việt |
|---|---|---|
| `healthy` | healthy | lúa khỏe |
| `blast` | rice blast | bệnh đạo ôn |
| `brown_spot` | brown spot | bệnh đốm nâu |
| `bacterial_blight` | bacterial leaf blight | bệnh bạc lá (cháy bìa lá) |
| `nitrogen_deficiency` | nitrogen deficiency | thiếu đạm |

**Columns · Các cột:** `id, image` (path inside the split folder · đường dẫn trong thư mục của tập)`, text` (farmer's message · tin nhắn của nông dân)`,
province, soil_type, variety, season, days_after_sowing, humidity_7d, temp_max_7d, temp_min_7d, rainfall_7d_mm, storm_last_3d,
nitrogen_kg_ha` (has NaN · có giá trị thiếu)`, field_area_ha, label` (target, train only · nhãn, chỉ có ở train).

*Inspired by · Lấy cảm hứng từ:* PetFinder.my – Pawpularity Contest (Kaggle, 2021), Shopee – Price Match Guarantee (Kaggle, 2021)
and · và Plant Pathology 2020/2021 (Kaggle, FGVC7/FGVC8).

🇬🇧 This baseline runs end-to-end and submits a result. Beating it is up to you.
🇻🇳 Baseline này chạy trọn vẹn và nộp được kết quả. Vượt qua nó là việc của bạn.

## 1. Setup and Imports · Cài đặt và thư viện

In [ ]:
import os
import re
import shutil
import unicodedata
import zipfile
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from PIL import Image
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

## 2. Configuration and Hyperparameters · Cấu hình và siêu tham số
🇬🇧 Nothing to fill in here: the data downloads itself, and your team comes from your account on the site when you submit (section 7).
🇻🇳 Không cần điền gì ở đây: dữ liệu tự tải về, và đội của bạn được lấy từ tài khoản trên trang web khi nộp bài (phần 7).


In [ ]:
# ---- Data / Dữ liệu ----
# The dataset downloads itself from GitHub straight onto this Colab session's own
# disk. No Google Drive and no mounting: reading thousands of small files over a
# Drive mount is what made the old flow slow, and local disk is several times
# faster per epoch.
# Bộ dữ liệu tự tải từ GitHub vào ổ đĩa của phiên Colab này. Không cần Drive.
TASK    = "2B"
FOLDER  = "2B_rice_multimodal"
RELEASE = "https://github.com/ftulabs/olympiad-datasets/releases/download/warmup-2"

# Already have the folder? Point this at the parent of 2B_rice_multimodal/ and nothing is
# downloaded. / Đã có sẵn thư mục? Trỏ vào thư mục cha của 2B_rice_multimodal/.
ROOT_OVERRIDE = ""

import os, subprocess, time

ROOT = ROOT_OVERRIDE or "/content/olympiad"
if not os.path.isdir(f"{ROOT}/{FOLDER}/dataset"):
    os.makedirs(ROOT, exist_ok=True)
    zip_path = f"{ROOT}/{FOLDER}.zip"
    if not os.path.exists(zip_path):
        print(f"Downloading {FOLDER}.zip / Đang tải ...", flush=True)
        t0 = time.time()
        subprocess.run(["curl", "-fsSL", "--retry", "3", "-o", zip_path,
                        f"{RELEASE}/{FOLDER}.zip"], check=True)
        print(f"  {os.path.getsize(zip_path)/1e6:.0f} MB in {time.time()-t0:.0f}s", flush=True)
    print("Extracting / Đang giải nén ...", flush=True)
    subprocess.run(["unzip", "-nq", zip_path, "-d", ROOT], check=True)

if not os.path.isdir(f"{ROOT}/{FOLDER}/dataset"):
    raise FileNotFoundError(
        f"{ROOT}/{FOLDER}/dataset is missing after the download. Re-run this cell. "
        f"/ Thiếu dữ liệu sau khi tải. Hãy chạy lại ô này."
    )
print(f"Data root / Thư mục dữ liệu: {ROOT}")

DATA_DIR = f"{ROOT}/2B_rice_multimodal/dataset"
TRAIN_DIR = f"{DATA_DIR}/train"
PUBLIC_DIR = f"{DATA_DIR}/public_test"
PRIVATE_DIR = f"{DATA_DIR}/private_test"
RESULTS_DIR = f"{ROOT}/results"  # shared submission folder / thư mục nộp bài chung
SAVE_DIR = "./checkpoints"

# ---- Hyperparameters / Siêu tham số ----
BATCH_SIZE = 64
EPOCHS = 6
LR = 1e-3
EMB_DIM = 32     # word-embedding size / kích thước embedding của từ
MAX_LEN = 48     # max words per message / số từ tối đa mỗi tin nhắn
HIDDEN = 64
SEED = 42

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

os.makedirs(SAVE_DIR, exist_ok=True)
np.random.seed(SEED)
torch.manual_seed(SEED)

## 3. Data Loading and Preprocessing · Đọc và tiền xử lý dữ liệu

In [ ]:
train_df = pd.read_csv(f"{TRAIN_DIR}/train.csv")
public_df = pd.read_csv(f"{PUBLIC_DIR}/public_test.csv")
private_df = pd.read_csv(f"{PRIVATE_DIR}/private_test.csv")
print(f"Train: {len(train_df)} | Public test: {len(public_df)} | Private test: {len(private_df)}")

TARGET = "label"
CLASSES = ["healthy", "blast", "brown_spot", "bacterial_blight", "nitrogen_deficiency"]
NUMERIC = ["days_after_sowing", "humidity_7d", "temp_max_7d", "temp_min_7d", "rainfall_7d_mm",
           "storm_last_3d", "nitrogen_kg_ha", "field_area_ha"]  # numeric columns only / chỉ dùng cột số

# Stratified split keeps the class mix equal in train and validation
# Chia phân tầng để tỷ lệ các lớp ở train và validation như nhau
tr_df, val_df = train_test_split(train_df, test_size=0.2, stratify=train_df[TARGET], random_state=SEED)


# ---- (a) Images / Ảnh: keep as uint8 in memory (small) / giữ dạng uint8 trong bộ nhớ (nhỏ) ----
def load_images(df, split_dir):
    arrays = [np.asarray(Image.open(f"{split_dir}/{p}").convert("RGB")) for p in df["image"]]
    return torch.tensor(np.stack(arrays)).permute(0, 3, 1, 2).contiguous()  # (N, 3, 96, 96)


# ---- (b) Text / Văn bản: whitespace tokens, vocabulary from the training part ----
# ---- tách từ theo khoảng trắng, bộ từ vựng lấy từ phần train ----
def tokenize(text):
    return str(text).lower().split()


counts = pd.Series([w for t in tr_df["text"] for w in tokenize(t)]).value_counts()
vocab = {w: i + 2 for i, w in enumerate(counts[counts >= 2].index)}  # 0 = padding, 1 = unknown / chưa gặp


def encode_text(df):
    ids = np.zeros((len(df), MAX_LEN), dtype=np.int64)
    for i, t in enumerate(df["text"]):
        tok = [vocab.get(w, 1) for w in tokenize(t)][:MAX_LEN]
        ids[i, :len(tok)] = tok
    return torch.tensor(ids)


# ---- (c) Tabular / Dữ liệu bảng: standardise numeric columns, NaN -> 0 ----
# ---- chuẩn hóa các cột số, NaN -> 0 ----
mean, std = tr_df[NUMERIC].mean(), tr_df[NUMERIC].std()


def encode_tab(df):
    x = ((df[NUMERIC] - mean) / std).fillna(0.0)
    return torch.tensor(x.values, dtype=torch.float32)


def make_dataset(df, split_dir, with_label=True):
    tensors = [load_images(df, split_dir), encode_text(df), encode_tab(df)]
    if with_label:
        tensors.append(torch.tensor(df[TARGET].map(CLASSES.index).values))
    return TensorDataset(*tensors)


train_ds = make_dataset(tr_df, TRAIN_DIR)
val_ds = make_dataset(val_df, TRAIN_DIR)
public_ds = make_dataset(public_df, PUBLIC_DIR, with_label=False)
private_ds = make_dataset(private_df, PRIVATE_DIR, with_label=False)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=256)
public_loader = DataLoader(public_ds, batch_size=256)
private_loader = DataLoader(private_ds, batch_size=256)
print(f"Train: {len(train_ds)} | Validation: {len(val_ds)} | Vocabulary: {len(vocab) + 2} words")
train_df.head()

## 4. Model Architecture · Kiến trúc mô hình

In [ ]:
class MultiModalNet(nn.Module):
    """Image CNN + text bag-of-words + tabular MLP, concatenated (late fusion).
    CNN cho ảnh + túi từ cho văn bản + MLP cho dữ liệu bảng, ghép nối ở cuối (late fusion)."""

    def __init__(self, vocab_size, n_tab, n_classes=len(CLASSES)):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),   # 96 -> 48
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),  # 48 -> 24
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(),
            nn.AdaptiveAvgPool2d(1), nn.Flatten(),                        # -> 64
        )
        self.text = nn.EmbeddingBag(vocab_size, EMB_DIM, mode="mean", padding_idx=0)
        self.tab = nn.Sequential(nn.Linear(n_tab, 32), nn.ReLU())
        self.head = nn.Sequential(nn.Linear(64 + EMB_DIM + 32, HIDDEN), nn.ReLU(), nn.Linear(HIDDEN, n_classes))

    def forward(self, img, tokens, tab):
        img = img.float() / 255.0  # uint8 -> [0, 1]
        feats = torch.cat([self.cnn(img), self.text(tokens), self.tab(tab)], dim=1)
        return self.head(feats)


model = MultiModalNet(len(vocab) + 2, len(NUMERIC)).to(DEVICE)
print(model)
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters()):,}")

## 5. Training · Huấn luyện

In [ ]:
@torch.no_grad()
def predict(model, loader):
    """Return predicted class indices. / Trả về chỉ số lớp dự đoán."""
    model.eval()
    preds = []
    for batch in loader:
        img, tokens, tab = (t.to(DEVICE) for t in batch[:3])
        preds.append(model(img, tokens, tab).argmax(1).cpu())
    return torch.cat(preds).numpy()


y_val = val_ds.tensors[3].numpy()
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

best_f1 = -1.0
for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0.0
    for img, tokens, tab, yb in train_loader:
        img, tokens, tab, yb = img.to(DEVICE), tokens.to(DEVICE), tab.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(img, tokens, tab), yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(yb)

    val_f1 = f1_score(y_val, predict(model, val_loader), average="macro")
    msg = f"Epoch {epoch:02d} | Loss={total_loss / len(train_ds):.4f} | Val macro-F1={val_f1:.4f}"
    if val_f1 > best_f1:
        best_f1 = val_f1
        torch.save(model.state_dict(), os.path.join(SAVE_DIR, "best_model.pt"))
        msg += "  ✓ best model saved"
    print(msg)

print(f"\nBest validation macro-F1: {best_f1:.4f}")

## 6. Predict Public & Private Test · Dự đoán tập Public & Private
🇬🇧 Creates `public_submission.csv` / `private_submission.csv` (+ `.zip`), like the official competition.
🇻🇳 Tạo `public_submission.csv` / `private_submission.csv` (+ `.zip`), giống định dạng cuộc thi chính thức.

In [ ]:
def export(ids, preds, csv_path):
    """Save predictions as <id, TARGET> CSV and zip it. / Lưu dự đoán ra CSV và nén zip."""
    pd.DataFrame({"id": ids, TARGET: preds}).to_csv(csv_path, index=False)
    zip_path = csv_path.replace(".csv", ".zip")
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        zf.write(csv_path, arcname=os.path.basename(csv_path))
    print(f"Saved {csv_path} and {zip_path}")


model.load_state_dict(torch.load(os.path.join(SAVE_DIR, "best_model.pt"), map_location=DEVICE))
export(public_df["id"], np.array(CLASSES)[predict(model, public_loader)], "public_submission.csv")
export(private_df["id"], np.array(CLASSES)[predict(model, private_loader)], "private_submission.csv")
pd.read_csv("public_submission.csv").head()

## 7. Submit · Nộp bài
🇬🇧 Checks the format, then copies both files to the shared results folder
`results/<task>/<team>/<timestamp>/`. You may submit several times; each run creates a new entry.
🇻🇳 Kiểm tra định dạng, sau đó sao chép hai file vào thư mục kết quả chung
`results/<task>/<team>/<thời điểm>/`. Có thể nộp nhiều lần; mỗi lần chạy tạo một lượt nộp mới.

In [ ]:
def check_submission(csv_path, test_df):
    """Validate the file format before submitting. / Kiểm tra định dạng file trước khi nộp."""
    sub = pd.read_csv(csv_path)
    assert list(sub.columns) == ["id", TARGET], f"{csv_path}: columns must be ['id', '{TARGET}']"
    assert len(sub) == len(test_df) and set(sub["id"]) == set(test_df["id"]), f"{csv_path}: ids do not match the test set"
    assert sub[TARGET].notna().all(), f"{csv_path}: contains missing values / có giá trị bị thiếu"
    assert sub[TARGET].isin(CLASSES).all(), f"{csv_path}: labels must be one of / nhãn phải thuộc {CLASSES}"


check_submission("public_submission.csv", public_df)
check_submission("private_submission.csv", private_df)


In [ ]:
# ===================================================================
#  Olympiad AI - submit.  Paste at the END of your notebook, then run.
#  No key to copy: the first run prints a code you enter on the site.
# ===================================================================
BASE    = "https://seleccion.ftds.online"
PROBLEM = "2B"          # which problem this notebook is for
CONTEST = "warmup-2"    # this notebook is for Warmup Contest 2
PUBLIC  = "public_submission.csv"
PRIVATE = "private_submission.csv"
# -------------------------------------------------------------------
import os, time, requests

token = globals().get("_OLY_TOKEN")
if not token:
    start = requests.post(f"{BASE}/api/pair/start", timeout=30).json()
    print(f"\n  1. open   {start['enter_at']}")
    print(f"  2. enter  {start['code']}\n  waiting...", flush=True)
    for _ in range(int(start["expires_in"]) // 3):
        time.sleep(3)
        got = requests.get(f"{BASE}/api/pair/poll",
                           params={"code": start["code"]}, timeout=30).json()
        if got.get("ready"):
            token = globals()["_OLY_TOKEN"] = got["token"]
            print("  paired ✓ (this notebook stays paired for the session)")
            break
    else:
        raise SystemExit("pairing timed out - run this cell again")

missing = [f for f in (PUBLIC, PRIVATE) if not os.path.exists(f)]
if PUBLIC in missing:
    raise SystemExit(f"{PUBLIC} not found - run the cell that writes it first")

files = {"public": open(PUBLIC, "rb")}
if PRIVATE not in missing:
    files["private"] = open(PRIVATE, "rb")
try:
    r = requests.post(f"{BASE}/api/submit", headers={"X-Oly-Token": token},
                      data={"problem": PROBLEM, "contest": CONTEST},
                      files=files, timeout=600)
finally:
    for f in files.values():
        f.close()

try:
    d = r.json()
except ValueError:
    raise SystemExit(f"server {r.status_code}: {r.text[:300]}")

if r.status_code == 429:
    print(f"⏳ cooldown - wait {d.get('retry_after')}s and run again")
elif r.status_code == 401:
    globals().pop("_OLY_TOKEN", None)
    print(f"❌ {d.get('detail')}  (run this cell again to re-pair)")
elif r.status_code != 200:
    print(f"❌ {r.status_code}: {d.get('detail') or d.get('message')}")
elif d["public"]["status"] != "ok":
    # the grader's own message about your file - it says what to fix
    print(f"❌ {d['public']['status'].upper()}: {d['public']['message']}")
else:
    p, c = d["public"], (d.get("counting") or {})
    print(f"✅ {d['problem']} · {d['title']}")
    print(f"   {d['metric']} = {p['score']:.4f}   {p['tier'] or '(below the lowest tier)'}")
    print(f"   entry #{d['entries']}   ·   rank: {d['rank']}")
    if c.get("score") is not None and abs(c["score"] - p["score"]) > 1e-12:
        print(f"   counting entry: {c['score']:.4f} {c['tier']}")
    print(f"   {BASE}/")
